In [1]:
!pip install -q sacrebleu

import re
import math
from collections import Counter
import numpy as np
import pandas as pd
from sacrebleu.metrics import CHRF

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 4.4 MB/s eta 0:00:00


In [2]:
sentences_oare_path = "/kaggle/input/deep-past-initiative-machine-translation/Sentences_Oare_FirstWord_LinNum.csv"
train_path = "/kaggle/input/deep-past-initiative-machine-translation/train.csv"

sentences_oare = pd.read_csv(sentences_oare_path)
train = pd.read_csv(train_path)

In [3]:
train['transliteration_length'] = train.apply(lambda row: len(row['transliteration'].split(' ')), axis=1)
train['translation_length'] = train.apply(lambda row: len(row['translation'].split(' ')), axis=1)
train.head()

,oare_id,transliteration,translation,transliteration_length,translation_length
0,004a7dbd-57ce-46f8-9691-409be61c676e,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ...",50,70
1,0064939c-59b9-4448-a63d-34612af0a1b5,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,Itūr-ilī has received one textile of ordinary ...,6,8
2,0073f2c0-524c-4bbf-915a-8c1772a4fb98,TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR,... he did not give you a textile. He returned...,7,16
3,009fb838-8038-42bc-ad34-5f795b3840ee,KIŠIB šu-(d)EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...,"Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū...",23,36
4,00aa1c55-c80c-4346-a159-73ad43ab0ff7,um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...,From Šukkutum to Ištar-lamassī and Nitahšušar:...,77,96


In [4]:
test = train.merge(sentences_oare, left_on='oare_id', right_on='text_uuid', suffixes=('_train', '_sentcs'))
print(test.shape, len(test['oare_id'].unique()))
test.to_csv("sentence_match_full.csv", index=False)
test.head()

(1213, 17) 253


,oare_id,transliteration,translation_train,transliteration_length,translation_length,display_name,text_uuid,sentence_uuid,sentence_obj_in_text,translation_sentcs,first_word_transcription,first_word_spelling,first_word_number,first_word_obj_in_text,line_number,side,column
0,004a7dbd-57ce-46f8-9691-409be61c676e,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ...",50,70,Kt 94/k 1122 envelope (AKT 6c 705),004a7dbd-57ce-46f8-9691-409be61c676e,6e455ba9-94e5-4d7a-a00a-b2ae79e4a2fb,3,"Seals of Mannum-bālum-Aššur s. ?, Šū-Enlil s. ...",kunuk,KIŠIB,1,5,1.0,1,1
1,004a7dbd-57ce-46f8-9691-409be61c676e,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ...",50,70,Kt 94/k 1122 envelope (AKT 6c 705),004a7dbd-57ce-46f8-9691-409be61c676e,a1beac04-7b5e-4692-9b88-0f1fb37cbf2b,26,Puzur-Aššur s. Attaya owes Al-aḫum 1/3 mina 2 ...,manā,ma-na,14,31,5.0,1,1
2,004a7dbd-57ce-46f8-9691-409be61c676e,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ...",50,70,Kt 94/k 1122 envelope (AKT 6c 705),004a7dbd-57ce-46f8-9691-409be61c676e,8b89b46d-98d5-4398-9e21-98778613b015,48,He will pay fourteen weeks from the ḫamuštum o...,ištu,iš-tù,25,50,8.0,1,1
3,004a7dbd-57ce-46f8-9691-409be61c676e,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ...",50,70,Kt 94/k 1122 envelope (AKT 6c 705),004a7dbd-57ce-46f8-9691-409be61c676e,0f0afbb8-98a9-497d-bf29-4ff0a838c5cf,71,"If he does not pay (on time), he will add 1.5 ...",šumma,šu-ma,39,73,13.0,1,1
4,009fb838-8038-42bc-ad34-5f795b3840ee,KIŠIB šu-(d)EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...,"Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū...",23,36,Kt 94/k 1041A envelope (AKT 6a 43),009fb838-8038-42bc-ad34-5f795b3840ee,17b1caa0-3d8d-4e03-94c2-764ba5450c2b,2,"Seal of Šu-Illil son of Šu-kūbum, seal of Şilū...",kunuk,KIŠIB,1,4,1.0,1,1


In [5]:
df = test.copy()

chrf = CHRF(word_order=2)

def split_translation_clauses(text):
    # conservative clause splitting
    return [t.strip() for t in text.replace(";", ".").split(".") if t.strip()]

aligned_pairs = []

for oare_id, g in df.groupby("oare_id"):
    g = g.sort_values("first_word_number")

    # tokenized transliteration (full tablet)
    full_translit_tokens = g.iloc[0]["transliteration"].split()
    translation_full = g.iloc[0]["translation_train"]
    translation_clauses = split_translation_clauses(translation_full)

    # sentence boundaries
    word_starts = g["first_word_number"].tolist()
    word_starts.append(len(full_translit_tokens) + 1)

    for i, row in g.iterrows():
        start = row["first_word_number"] - 1
        end = word_starts[g.index.get_loc(i) + 1] - 1

        translit_sentence = " ".join(full_translit_tokens[start:end]).strip()
        anchor = str(row["translation_sentcs"])

        if not translit_sentence or not anchor:
            continue

        # fuzzy match translation clauses
        scores = []
        for j, clause in enumerate(translation_clauses):
            score = chrf.sentence_score(anchor, [clause]).score
            scores.append((score, j))

        if not scores:
            continue

        best_idx = max(scores)[1]

        # try expanding window
        candidate = translation_clauses[best_idx]
        if best_idx + 1 < len(translation_clauses):
            expanded = candidate + ", " + translation_clauses[best_idx + 1]
            if chrf.sentence_score(anchor, [expanded]).score > \
               chrf.sentence_score(anchor, [candidate]).score:
                candidate = expanded

        aligned_pairs.append((oare_id, translit_sentence, candidate, anchor))

print(f"Aligned sentence pairs: {len(aligned_pairs)}")
aligned_pairs[:3]

Aligned sentence pairs: 1052


[('004a7dbd-57ce-46f8-9691-409be61c676e',
  'KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠIB šu-(d)EN.LÍL DUMU ma-nu-ki-a-šur KIŠIB MAN-a-šur DUMU a-ta-a 0.33333',
  'Seal of Mannum-balum-Aššur son of Ṣilli-Adad, seal of Šu-Illil son of Mannum-kī-Aššur, seal of Puzur-Aššur son of Ataya',
  'Seals of Mannum-bālum-Aššur s. ?, Šū-Enlil s. ?, Puzur-Aššur s. Attaya'),
 ('004a7dbd-57ce-46f8-9691-409be61c676e',
  'ma-na 2 GÍN KÙ.BABBAR SIG₅ i-ṣé-er PUZUR₄-a-šur DUMU a-ta-a a-lá-ḫu-um i-šu',
  'Puzur-Aššur son of Ataya owes 22 shekels of good silver to Ali-ahum',
  'Puzur-Aššur s. Attaya owes Al-aḫum 1/3 mina 2 shekels fine silver.'),
 ('004a7dbd-57ce-46f8-9691-409be61c676e',
  'iš-tù ḫa-muš-tim ša ì-lí-dan ITU.KAM ša ke-na-tim li-mu-um e-na-sú-in a-na ITU 14 ḫa-am-ša-tim i-ša-qal',
  'Reckoned from the week of Ilī-dan, month of Ša-kēnātim, in the eponymy of Enna-Suen, he will pay in 14 weeks',
  'He will pay fourteen weeks from the ḫamuštum of Ilīdan month of ša kēnātim eponymate Enna-Suen. '

In [6]:
res = pd.DataFrame(aligned_pairs, columns=["oare_id", "translit_sent", "translation_sent_train", "translation_anchor"])
res.to_csv("sentence_alignment.csv", index=False)